In [1]:
# instalasi depedensi GRPO
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl>=0.15.0" peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-435kh3eo/unsloth_b64735495b83446c9202de706b38136f
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-435kh3eo/unsloth_b64735495b83446c9202de706b38136f
  Resolved https://github.com/unslothai/unsloth.git to commit 2aef1a23cb92922cb22d3e484b1e67f241c31c3e
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 109.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 80.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 17.4 MB/s eta 0:00:00
   

In [2]:
# load model & setup LORA
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

print("[INFO] Memuat Model Llama-3 untuk Operasi GRPO...")

# unduh Model Llama-3 versi Instruct
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Pasang LoRA Adapters
print("[INFO] Memasang LoRA Adapters...")
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

print("[INFO] Model Llama-3 dan LoRA Adapters berhasil diaktifkan!")



[INFO] Memuat Model Llama-3 untuk Operasi GRPO...
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


<string>:42: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/345 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3-8b-Instruct-bnb-4bit as a legacy tokenizer.


[INFO] Memasang LoRA Adapters...


Unsloth 2026.6.9 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


[INFO] Model Llama-3 dan LoRA Adapters berhasil diaktifkan!


In [3]:
# definisi 4 fungsi reward GRPO
import re

def ekstrak_jawaban(teks):
  # fungsi untuk memisahkan teks <think> dan teks jawaban akhir
  if isinstance(teks, list):
    teks = teks[0]['content']

  # Ambil teks di dalam <think>
  think_match = re.search(r"<think>(.*?)</think>", teks, re.DOTALL)
  teks_mikir = think_match.group(1) if think_match else ""

  # Ambil teks setelah </think>
  teks_jawaban = teks.split("</think>")[-1].strip() if "</think>" in teks else teks
  return teks_mikir, teks_jawaban

# 1. Format Reward
# memberi poin 1.0 jika model menggunakan tag <think>...</think> dengan benar.
def format_reward(completions, **kwargs):
  rewards = []
  for c in completions:
    teks = c[0]['content'] if isinstance(c, list) else c
    if "<think>" in teks and "</think>" in teks:
      rewards.append(1.0)
    else:
      rewards.append(0.0)
  return rewards

# 2. Reasoning Length Reward
# mendorong model agar tidak malas. semakin panjang ia berpikir, semakin besar poinnya.
def reasoning_length_reward(completions, **kwargs):
  rewards = []
  for c in completions:
    teks_mikir, _ = ekstrak_jawaban(c)
    panjang_karakter = len(teks_mikir)

    if panjang_karakter > 300:
      rewards.append(1.5)
    elif panjang_karakter > 50:
      rewards.append(0.5)
    else:
      rewards.append(0.0)
  return rewards

# 3. Correctness Reward
# memastikan model menyimpulkna hukum dengan tepat
def correctness_reward(completions, prompts, **kwargs):
  rewards = []
  for c, p in zip(completions, prompts):
    _, teks_jawaban = ekstrak_jawaban(c)
    jawaban_lower = teks_jawaban.lower()

    if "berhak" in jawaban_lower or "ya" in jawaban_lower or "mendapat" in jawaban_lower:
      rewards.append(2.0)
    else:
      rewards.append(-2.0)
  return rewards

# 4. Language Reward
# Menghukum model jika dia "Halusinasi" mencampuradukkan  dengan bahasa Inggris
def language_reward(completions, **kwargs):
  rewards = []
  for c in completions:
    _, teks_jawaban = ekstrak_jawaban(c)

    kata_inggris = [" the ", " is ", " and ", " overtime ", " employee "]
    if any(eng in teks_jawaban.lower() for eng in kata_inggris):
      rewards.append(-0.5)
    else:
      rewards.append(0.5)
  return rewards

print("[INFO] 4 fungsi reward GRPO sudah didefinisikan!")

[INFO] 4 fungsi reward GRPO sudah didefinisikan!


In [7]:
# Dataset & Setup GRPO trainer
from trl import GRPOConfig, GRPOTrainer
from datasets import Dataset

print("[INFO] Menyiapkan arena pelatihan (Training Dataset)...")

dataset_latihan = Dataset.from_dict({
    "prompt": [
        [
            {"role": "system", "content": "Kamu adalah pakar hukum Indonesia. Berikan jawaban dengan format <think> [proses berpikir] </think> [jawaban akhir]."},
            {"role": "user", "content": "Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang lembur?"}
        ]
    ] * 8  # dibuat 8 kali agar model berlatih cukup dalam satu iterasi
})

print("[INFO] Mengatur Hyperparameter GRPO Trainer...")
training_args = GRPOConfig(
    output_dir= "Legal_GRPO_Output",
    learning_rate = 5e-6,
    lr_scheduler_type = "cosine",
    logging_steps = 1,
    max_steps = 10,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 2,
    max_prompt_length = 256,
    max_completion_length = 512,
    num_generations = 4,
    optim = "adamw_8bit",
    report_to = "none"
)

print("[INFO] Menggabungkan Model, Dataset, dan 4 fungsi reward GRPO...")
trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        format_reward,
        reasoning_length_reward,
        correctness_reward,
        language_reward,
    ],
    args = training_args,
    train_dataset = dataset_latihan,
)

print("[INFO] GRPO siap!")


[INFO] Menyiapkan arena pelatihan (Training Dataset)...
[INFO] Mengatur Hyperparameter GRPO Trainer...
Unsloth: We now expect `per_device_train_batch_size` * `gradient_accumulation_steps` * `world_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 4
[INFO] Menggabungkan Model, Dataset, dan 4 fungsi reward GRPO...
[INFO] GRPO siap!


In [8]:
# training GRPO
print("[INFO] Memulai proses Pelatihan (Training) GRPO...")

trainer.train()

# simpan hasil adapter setelah selesai
print("\n [INFO] Pelatihan selesai! Menyimpan hasil adapatasi otak model...")
model.save_pretrained("Legal_GRPO_Output")
print("[INFO] Model tersimpan di folder 'Legal_GRPO_Output'!")

[INFO] Memulai proses Pelatihan (Training) GRPO...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 8 | Num Epochs = 3 | Total steps = 10
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
Passing `generation_config` together with generation-related arguments=({'disable_compile', 'pad_token_id', 'cache_implementation'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=512) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarn

Unsloth: Will smartly offload gradients to save VRAM!


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: inductor::_alloc_from_pool (with implementation in ???): The output of this custom operator (1) must not also be an input to this custom operator and (2) may not alias any inputs to this custom operator or other returns. The most common way to trigger this error is if we have y = custom_op(x) and y and x are the same Tensor. Please instead return a clone of the offending output tensor(s) (e.g. return x.clone()) or refactor the custom operator to not return y. This is deprecated and will become an error in PyTorch 2.12.
  return fn(*args, **kwargs)


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / format_reward / mean,rewards / format_reward / std,rewards / reasoning_length_reward / mean,rewards / reasoning_length_reward / std,rewards / correctness_reward / mean,rewards / correctness_reward / std,rewards / language_reward / mean,rewards / language_reward / std
1,-0.149311,2.562500,2.621737,280.375000,159.000000,413.000000,0.000000,280.375000,159.000000,413.000000,0.000009,0.625000,0.517549,0.687500,0.703943,1.000000,1.851640,0.250000,0.462910
2,-0.005122,2.875000,2.213154,302.125000,168.000000,376.000000,0.000000,302.125000,168.000000,376.000000,0.000008,0.750000,0.462910,0.875000,0.694365,1.000000,1.851640,0.250000,0.462910
3,0.045454,3.125000,1.927350,267.625000,186.000000,461.000000,0.000000,267.625000,186.000000,461.000000,0.000008,0.750000,0.462910,1.000000,0.707107,1.000000,1.851640,0.375000,0.353553
4,0.086660,3.375000,1.903723,274.875000,159.000000,500.000000,0.000000,274.875000,159.000000,500.000000,0.000010,0.750000,0.462910,0.750000,0.654654,1.500000,1.414214,0.375000,0.353553
5,-0.062535,4.375000,0.862372,255.500000,172.000000,361.000000,0.000000,255.500000,172.000000,361.000000,0.000013,0.875000,0.353553,1.000000,0.707107,2.000000,0.000000,0.500000,0.000000
6,0.154042,3.937500,1.312415,289.375000,139.000000,476.000000,0.000000,289.375000,139.000000,476.000000,0.000015,0.625000,0.517549,0.812500,0.752970,2.000000,0.000000,0.500000,0.000000
7,-0.038040,2.812500,1.145416,275.625000,181.000000,380.000000,0.000000,275.625000,181.000000,380.000000,0.000013,0.625000,0.517549,0.937500,0.776324,1.000000,1.851640,0.250000,0.462910
8,-0.100899,4.687500,0.625000,268.500000,127.000000,459.000000,0.000000,268.500000,127.000000,459.000000,0.000019,0.875000,0.353553,1.312500,0.530330,2.000000,0.000000,0.500000,0.000000
9,-0.040813,3.000000,2.386270,300.625000,127.000000,512.000000,0.125000,270.428589,127.000000,412.000000,0.000017,0.500000,0.534522,0.625000,0.744024,1.500000,1.414214,0.375000,0.353553
10,0.055833,4.312500,0.765388,296.125000,230.000000,475.000000,0.000000,296.125000,230.000000,475.000000,0.000018,0.875000,0.353553,1.062500,0.623212,2.000000,0.000000,0.375000,0.353553


Both `max_new_tokens` (=512) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=51


 [INFO] Pelatihan selesai! Menyimpan hasil adapatasi otak model...
[INFO] Model tersimpan di folder 'Legal_GRPO_Output'!


In [9]:
# uji coba GRPO Inference
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

pertanyaan = "Saya staf Admin, kemarin lembur 5 jam untuk beresin laporan Keuangan tutup buku tahunan. Apakah saya berhak mendapatkan uang lembur?"

prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
Kamu adalah pakar hukum. Berikan jawaban dengan format <think> [proses berpikir] </think> [jawaban akhir].
<|eot_id|><|start_header_id|>user<|end_header_id|>
{pertanyaan}
<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# generate jawaban
outputs = model.generate(**inputs, max_new_tokens=512)
jawaban_lengkap = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("="*50)
print(" Hasil Penalaran Model GRPO: ")
print("="*50)
print(jawaban_lengkap.split("<|end_header_id|>")[-1].strip())


Both `max_new_tokens` (=512) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


 Hasil Penalaran Model GRPO: 
system
Kamu adalah pakar hukum. Berikan jawaban dengan format <think> [proses berpikir] </think> [jawaban akhir].
user
Saya staf Admin, kemarin lembur 5 jam untuk beresin laporan Keuangan tutup buku tahunan. Apakah saya berhak mendapatkan uang lembur?
assistant

<think> Berdasarkan peraturan perusahaan, apakah ada ketentuan yang mengatur tentang penggantian waktu lembur untuk staf Admin?</think>

<answer> Berdasarkan peraturan perusahaan, saya coba mencari informasi apakah ada ketentuan yang mengatur tentang penggantian waktu lembur untuk staf Admin. Setelah mencari, saya menemukan bahwa perusahaan memiliki ketentuan bahwa staf Admin yang melakukan lembur di luar jam kerja dapat diberikan gaji tambahan sebagai kompensasi. Oleh karena itu, saya berhak mendapatkan uang lembur karena saya telah melakukan lembur 5 jam untuk beresin laporan Keuangan tutup buku tahunan.
